# Augmented ODEs: terms and hidden states

When a mechanistic model can't fit the data and the residuals are structured, the question is *what is missing, and where*. MxlPy answers it inside the regular toolchain, without translating the model to JAX:

- a **term** rewrites a variable's derivative or a reaction's rate through an explicit `mechanism`;
- a **hidden-state block** adds latent states `h` driven by a small neural network, `dh/dt = nn_h(x, h)`, and corrects each target with `W h`. The hidden states give the model *memory*: delays, unmodelled slow processes, history dependence.

Both are lowered to symbolic equations, so every integrator (including `Native`), fits, forward and adjoint sensitivities, code generation and `.mxl.json` files work with them. The contributions of the terms show **which** variables or reactions miss dynamics, and **when**.

In [ ]:
import numpy as np
import pandas as pd

from mxlpy import Model, Simulator, fit, fns, plot


## A model that misses a delay

The data come from `S → X → P`; our mechanistic model only knows `S → P`.

In [ ]:
def truth() -> Model:
    return (
        Model()
        .add_parameters({"k": 1.0, "kx": 0.7})
        .add_variables({"S": 1.0, "X": 0.0, "P": 0.0})
        .add_reaction("v1", fns.mass_action_1s, args=["S", "k"], stoichiometry={"S": -1, "X": 1})
        .add_reaction("v2", fns.mass_action_1s, args=["X", "kx"], stoichiometry={"X": -1, "P": 1})
    )


def mechanistic() -> Model:
    return (
        Model()
        .add_parameters({"k": 1.0})
        .add_variables({"S": 1.0, "P": 0.0})
        .add_reaction("v1", fns.mass_action_1s, args=["S", "k"], stoichiometry={"S": -1, "P": 1})
    )


time = np.linspace(0, 8, 17)
rng = np.random.default_rng(0)
res = Simulator(truth()).simulate_time_course(time).get_result().unwrap_or_err()
data = res.variables.loc[:, ["S", "P"]]
data = data + rng.normal(0, 0.01, size=data.shape)

mech_fit = fit.time_course(
    mechanistic(),
    p0={"k": 1.0},
    data=data,
    minimizer=fit.LocalScipyMinimizer(method="L-BFGS-B"),
    loss_fn=fit.GaussianNll(sigma=0.01),
).unwrap_or_err()
mech_fit.best_pars, mech_fit.loss

## Terms

`add_term(name, fn, args, target=..., mechanism=...)` works on variables and (kinetic models) reactions. The mechanism is required: `"additive"` (`base + term`), `"relative"` (`base · (1 + term)`) or any callable `(base, term) -> value`. A reaction term rewrites the rate *before* the stoichiometry fans it out, so it corrects every variable the reaction touches. Each term's contribution `Δ = mechanism(base, term) − base` is a named output, and `dx/dt = base + ΣΔ`. Terms never enter the stoichiometric matrix.

In [ ]:
m = mechanistic().add_parameter("f", 0.2)
m.add_term("slow_down", fns.constant, args=["f"], target="v1", mechanism=lambda base, term: base * (1 - term))
print(m.get_fluxes(), m.get_fluxes(corrected=True), m.get_term_contributions(), sep="\n\n")

## Hidden states

One hidden state with a small network, correcting the derivative of `P` relatively: `S` decays as modelled, only the appearance of `P` is delayed. (A correction of the reaction `v1` would change `S` and `P` alike and couldn't fit.) The weights are not parameters: `get_weight_names()` lists them and `get_addressable_names()` lists parameters and weights, the names fits, sensitivities and the simulator accept.

In [ ]:
model = mechanistic().add_hidden_states(
    "aug", n_hidden=1, layers=(4,), targets=["P"], mechanism="relative"
)
model.get_variable_names(), model.get_parameter_names(), len(model.get_weight_names())

## Training

`fit.augmented_time_course` starts at the mechanistic optimum: it scales the block's inputs and outputs along the mechanistic trajectory, fits the weights alone with Normal(0, 0.1) priors and a smooth penalty on the correction (so the mechanism keeps explaining what it can), then refines weights and the mechanistic parameters jointly.

In [ ]:
res = fit.augmented_time_course(
    model, data=data, p0=mech_fit.best_pars, sigma=0.01
).unwrap_or_err()
print(f"loss: mechanistic {mech_fit.loss:.1f}, augmented {res.loss:.1f}")

In [ ]:
sim = Simulator(res.model).simulate_time_course(np.linspace(0, 8, 101)).get_result().unwrap_or_err()
fig, ax = plot.lines(sim.variables.loc[:, ["S", "P"]])
ax.plot(data.index, data["P"], "o", color="C1")
ax.plot(data.index, data["S"], "o", color="C0")
ax.set(xlabel="time", ylabel="concentration")
plot.show()

The correction shows where and when the mechanism falls short: the production of `P` is slowed down early (the missing intermediate `X` is still filling up) and sped up later.

In [ ]:
fig, axs = plot.correction(res)
plot.show()

## Saving

Terms and blocks are saved to `.mxl.json` (spec_version 0.5), with the weights in a sidecar file. Blocks authored in mxlweb (`nn_blocks`, any mechanism) load the same way.

In [ ]:
import tempfile
from pathlib import Path

import mxlpy

with tempfile.TemporaryDirectory() as tmp:
    mxlpy.save(res.model, Path(tmp) / "augmented.mxl.json")
    loaded = mxlpy.load(Path(tmp) / "augmented.mxl.json")
loaded.get_raw_hidden_states()["aug"].n_hidden

## Notes

- With the readout `W = 0` the derivatives of the mechanistic variables are exactly those of the mechanistic model. Simulations agree to the solver's tolerance, as the hidden states take part in its step-size control.
- `model.remove_hidden_states("aug")` restores the mechanistic model, e.g. to try `n_hidden = 1, 2, …`.
- Gradients w.r.t. many weights are cheapest with adjoints (`mxlpy.adjoint_sensitivities`), chosen automatically for large problems.